# Inference Service

https://ollama.com/download

Before running it, start Ollama and pull a small model:

```bash
ollama serve
ollama pull qwen3:0.6b
```

In [46]:
import time
from statistics import mean

import openai

In [47]:
MODEL = "qwen3:0.6b"
BASE_URL = "http://localhost:11434/v1"

client = openai.Client(api_key="ollama", base_url=BASE_URL)

## 1. One Basic Request

First, check that the local inference service works.

In [48]:
result = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Hi."}],
)

print(result.choices[0].message.content)

Hello! How can I assist you today? 😊


## 2. Helper: Token Usage

In [49]:
def extract_usage(response):
    usage = getattr(response, "usage", None)
    if usage is None:
        return {
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
        }

    return {
        "prompt_tokens": getattr(usage, "prompt_tokens", None),
        "completion_tokens": getattr(usage, "completion_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
    }


def spent_tokens(response):
    usage = extract_usage(response)
    return usage["total_tokens"]

In [50]:
print(extract_usage(result))

{'prompt_tokens': 12, 'completion_tokens': 89, 'total_tokens': 101}


## 3. Helper: E2E Latency

In [51]:
def measure_e2e_latency(prompt, *, model=MODEL, max_tokens=500, temperature=1.0):
    messages = [{"role": "user", "content": prompt}]

    start = time.perf_counter()
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
        stream=False,
    )
    e2e_latency = time.perf_counter() - start
    usage = extract_usage(response)

    return {
        "response": response,
        "text": response.choices[0].message.content,
        "e2e_latency": e2e_latency,
        "prompt_tokens": usage["prompt_tokens"],
        "completion_tokens": usage["completion_tokens"],
        "total_tokens": usage["total_tokens"],
    }

In [53]:
e2e = measure_e2e_latency(
    "Explain KV cache in one sentence.",
    max_tokens=500,
)

print("E2E latency:", round(e2e["e2e_latency"], 3), "sec")
print("Prompt tokens:", e2e["prompt_tokens"])
print("Completion tokens:", e2e["completion_tokens"])
print("Total tokens:", e2e["total_tokens"])
print("Output:", e2e["text"])

E2E latency: 0.981 sec
Prompt tokens: 18
Completion tokens: 260
Total tokens: 278
Output: The KV cache in Keras is a memory-efficient mechanism used to store computational gradients, which helps in optimizing the model's performance by managing the computational load effectively.


## 4. Helper: TTFT With Streaming

In [62]:
def measure_ttft_and_streaming_latency(prompt, *, model=MODEL, max_tokens=64, temperature=0.0):
    messages = [{"role": "user", "content": prompt}]

    start = time.perf_counter()
    first_token_time = None
    output_chunks = []
    chunk_times = []

    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
        stream=True,
    )

    prev = start
    for chunk in stream:
        now = time.perf_counter()
        delta = chunk.choices[0].delta
        content = getattr(delta, "content", None)

        if content:
            if first_token_time is None:
                first_token_time = now
            else:
                chunk_times.append(now - prev)
            prev = now
            output_chunks.append(content)

    end = time.perf_counter()

    ttft = first_token_time - start if first_token_time is not None else None
    e2e_latency = end - start

    return {
        "text": "".join(output_chunks),
        "ttft": ttft,
        "e2e_latency": e2e_latency,
        "num_stream_chunks": len(output_chunks),
        "chunk_times": chunk_times,
    }

In [63]:
stream_metrics = measure_ttft_and_streaming_latency(
    "Explain continuous batching in one short paragraph.",
    max_tokens=500,
)

print("TTFT:", round(stream_metrics["ttft"], 3) if stream_metrics["ttft"] is not None else None, "sec")
print("E2E latency:", round(stream_metrics["e2e_latency"], 3), "sec")
print("Completion latency:", round(stream_metrics["e2e_latency"] - stream_metrics["ttft"], 3), "sec")
print("Stream chunks:", stream_metrics["num_stream_chunks"])
print("avg chunks times:", sum(stream_metrics["chunk_times"]) / len(stream_metrics["chunk_times"]))
print("Output:", stream_metrics["text"])

TTFT: 0.958 sec
E2E latency: 1.246 sec
Completion latency: 0.288 sec
Stream chunks: 92
avg chunks times: 0.003127803570580679
Output: Continuous batching refers to the process of mixing ingredients in a continuous, real-time manner, ensuring uniformity and efficiency. It involves blending ingredients like flour, water, and oil in a flowing stream, allowing for precise control over the mixture's properties. This method minimizes errors and ensures consistency, making it ideal for applications such as baking or manufacturing. By maintaining a steady flow, continuous batching enhances productivity and quality, making it a key technique in modern industrial processes.


## 5. Compare Prompt And Output Lengths


In [65]:
short_prompt = "Explain LLM inference in one sentence."
long_prompt = (
    "Explain LLM inference in one sentence. "
    + "This is filler text. " * 40
)

experiments = [
    ("short prompt, short output", short_prompt, 100),
    ("short prompt, longer output", short_prompt, 200),
    ("long prompt, short output", long_prompt, 100),
    ("long prompt, long output", long_prompt, 200),
]

rows = []
for name, prompt, max_tokens in experiments:
    metrics = measure_e2e_latency(prompt, max_tokens=max_tokens)
    rows.append({"name": name, **{k: v for k, v in metrics.items() if k != "response" and k != "text"}})

for row in rows:
    print(row["name"])
    print("  E2E latency:", round(row["e2e_latency"], 3), "sec")
    print("  Prompt tokens:", row["prompt_tokens"])
    print("  Completion tokens:", row["completion_tokens"])
    print("  Total tokens:", row["total_tokens"])
    print()

short prompt, short output
  E2E latency: 0.545 sec
  Prompt tokens: 19
  Completion tokens: 100
  Total tokens: 119

short prompt, longer output
  E2E latency: 0.71 sec
  Prompt tokens: 19
  Completion tokens: 200
  Total tokens: 219

long prompt, short output
  E2E latency: 0.407 sec
  Prompt tokens: 220
  Completion tokens: 100
  Total tokens: 320

long prompt, long output
  E2E latency: 0.621 sec
  Prompt tokens: 220
  Completion tokens: 170
  Total tokens: 390



## 6. Repeated Runs

Single measurements are noisy. Use repeated runs to compute simple averages.

In [66]:
def repeat_e2e(prompt, *, runs=3, max_tokens=64):
    measurements = []
    for _ in range(runs):
        measurements.append(measure_e2e_latency(prompt, max_tokens=max_tokens))

    return {
        "runs": runs,
        "avg_e2e_latency": mean(m["e2e_latency"] for m in measurements),
        "avg_completion_tokens": mean(m["completion_tokens"] for m in measurements if m["completion_tokens"] is not None),
        "measurements": measurements,
    }

summary = repeat_e2e("Explain rate limiting for an inference service.", runs=3, max_tokens=1000)
print("Average E2E latency:", round(summary["avg_e2e_latency"], 3), "sec")
print("Average completion tokens:", round(summary["avg_completion_tokens"], 1))

Average E2E latency: 2.569 sec
Average completion tokens: 770
